<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 12 · Deep Learning for Cross-Sectional and Panel Data

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We implement PyTorch MLPs for cross-sectional return prediction and autoencoder-based factor extraction.

### Getting Help While Using PyTorch
- **Appendix D** provides PyTorch syntax and training tips.
- **Appendix B** covers the pandas transformations used for features.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### Data Loading
We load the price panel once for this notebook.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Prepare Training Data

We extract a simple feature/label table from the price panel and convert it into PyTorch tensors, ready for mini‑batch training.

In [ ]:
panel = prices.ffill().pct_change().dropna()
features = panel.rolling(20).mean().stack().dropna()
labels = panel.shift(-1).stack().reindex(features.index).dropna()
features = features.loc[labels.index]
X = torch.from_numpy(features.values.astype(np.float32)).unsqueeze(-1)
y = torch.from_numpy(labels.values.astype(np.float32))
train_size = int(0.8 * len(X))
train_ds = TensorDataset(X[:train_size], y[:train_size])
val_ds = TensorDataset(X[train_size:], y[train_size:])
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)

## 2. Multi-Layer Perceptron

This section defines a compact MLP architecture with one hidden layer, ReLU activation, and dropout for regularization.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

### 2.1 Training Loop

We implement a minimal training loop with batches, loss computation, backpropagation, and validation loss tracking.

In [ ]:
def train(model, epochs=5):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_losses = []
            for xb, yb in val_loader:
                val_losses.append(loss_fn(model(xb), yb).item())
        print(f"Epoch {epoch+1}, val loss {np.mean(val_losses):.5f}")

train(model, epochs=5)

## 3. Autoencoder for Factor Extraction

Here we build an autoencoder whose bottleneck layer serves as a learned low‑dimensional factor representation of the inputs.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return latent, recon

auto = AutoEncoder()
opt = torch.optim.Adam(auto.parameters(), lr=1e-3)

### 3.1 Train Autoencoder

We train the autoencoder on the feature tensor and then inspect the latent codes that emerge from the encoder.

In [ ]:
for epoch in range(5):
    auto.train()
    epoch_loss = 0
    for xb, _ in train_loader:
        opt.zero_grad()
        _, recon = auto(xb)
        loss = loss_fn(recon.squeeze(-1), xb.squeeze(-1))
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}, loss {epoch_loss / len(train_loader):.5f}")

with torch.no_grad():
    latent_factors, _ = auto(X)
latent_factors[:5]

## 4. Exercises
### Exercise 1 – Hyperparameter Tuning
Vary hidden sizes/dropout and log validation loss per configuration.
<details><summary>Hint</summary>
Wrap the training loop in a function that accepts hidden units and dropout rate.
</details>

### Exercise 2 – GPU Check
Add a cell that uses <code>torch.cuda.is_available()</code> and moves tensors to GPU when available.
<details><summary>Hint</summary>
Call <code>to(device)</code> on both model and tensors.
</details>

### Exercise 3 – Factor Interpretation
Correlate latent factors with the original assets to label them (e.g., growth vs. defensive).
<details><summary>Hint</summary>
Convert <code>latent_factors.numpy()</code> to pandas and compute correlations with feature columns.
</details>


## 5. Takeaways for Chapter 12
- PyTorch lets us prototype deep models inside research notebooks.
- Autoencoders offer a data-driven angle on factor discovery.
- Even simple loops benefit from checkpoints, logging, and validation monitoring.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">